In [7]:
import cv2 as cv
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras import models, layers

In [8]:
DATASET_DIR = r"C:\Users\veerk\OneDrive\Desktop\Braille_To_Speech\Braille Dataset\Braille Dataset"
NUM_CLASSES = 26

In [9]:
img = cv.imread(r"C:\Users\veerk\OneDrive\Desktop\Braille_To_Speech\Braille Dataset\Braille Dataset\a1.JPG0whs.jpg")
IMG_HEIGHT, IMG_WIDTH = img.shape[:2]


In [10]:
def preprocess_image(img : np.ndarray) -> np.ndarray:    
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    blurred = cv.GaussianBlur(gray, (1, 1), 1)
    canny = cv.Canny(blurred, 120, 255)     
    _, thresh = cv.threshold(canny, 120, 255, cv.THRESH_BINARY_INV)
    return thresh

In [11]:
def resize_image(img : np.ndarray, scale_factor : int | float = 10) -> np.ndarray :
    dimensions = img.shape[0] * scale_factor, img.shape[1] * scale_factor
    return cv.resize(img, dimensions, interpolation=cv.INTER_AREA)

# image = cv.imread(r"C:\Users\veerk\OneDrive\Desktop\Braillle_To_Speech\Braille Dataset\Braille Dataset\c1.JPG11dim.jpg")
# cv.imshow("Image", resize_image(image))
# cv.imshow("Preprocessed", resize_image(preprocess_image(image)))
# cv.waitKey(0)
# cv.destroyAllWindows()

In [12]:
X, y = [], []
label_map = {chr(97 + i): i for i in range(26)}

for img_path in Path(DATASET_DIR).iterdir():
    img = cv.imread(img_path)
    preprocess = preprocess_image(img)
    X.append(preprocess)
    y.append(label_map[img_path.name[0]])

X = np.array(X)
y = np.array(y)

X = X.reshape(-1, IMG_HEIGHT, IMG_WIDTH, 1) / 255.0 
y = to_categorical(y, NUM_CLASSES)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 1)), 
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

c:\Users\veerk\OneDrive\Desktop\Braille_To_Speech\tfenv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 26)             │         3,354 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 227,098 (887.10 KB)

 Trainable params: 227,098 (887.10 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
#Training and Evaluation
model.fit(X_train, y_train, epochs=32, validation_data=(X_test, y_test))

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

Epoch 1/32
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.0738 - loss: 3.2197 - val_accuracy: 0.3718 - val_loss: 2.5644
Epoch 2/32
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4518 - loss: 2.1122 - val_accuracy: 0.6218 - val_loss: 1.5683
Epoch 3/32
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6310 - loss: 1.3487 - val_accuracy: 0.6346 - val_loss: 1.4227
Epoch 4/32
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6627 - loss: 1.1533 - val_accuracy: 0.6442 - val_loss: 1.3190
Epoch 5/32
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7193 - loss: 0.9623 - val_accuracy: 0.6378 - val_loss: 1.2764
Epoch 6/32
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7327 - loss: 0.9049 - val_accuracy: 0.7115 - val_loss: 1.1720
Epoch 7/32
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7717 - loss: 0.7691 - val_accuracy: 0.6987 - val_loss: 1.1535
Epoch 8/32
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8251 - loss: 0.6294 - val_accuracy: 0.7179 - val_